# Class XX: Final Project Topic
Goal of today's class:

1. Define Topic XX
2. Explore XX in different contexts
3. Third point

*Acknowledgement: This chapter relies on \_\_\_\_\_\_\_\_\_\_ from \_\_\_\_\_\_\_\_\_\_.*
__________

In [25]:
# import your packages! and whatever else that's basic that you need
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rc
rc('axes', fc='w')
rc('figure', fc='w')
rc('savefig', fc='w')
rc('axes', axisbelow=True)


# Social Contagion on Networks: Simple vs. Complex Contagion — A Deep Dive

**Audience:** PhD-level Network Science  
**Stack:** Python, `networkx`, `numpy`, `matplotlib`, `pandas`

## What & Why
This module bridges **literature**, **theory**, and **computation** for social contagion. We start with canonical works (Milgram; Watts–Strogatz; Newman; Christakis; Centola; KKT) to situate the field, then implement rigorous models to reveal how **structure** (clustering, shortcuts, degree heterogeneity) interacts with **mechanisms** (simple vs. complex contagion).



## Executive Summary
- **Simple contagion**: one effective exposure can trigger adoption (e.g., IC/SI-like spreading).  
- **Complex contagion**: requires **reinforcement** from multiple concordant exposures (e.g., Watts Threshold Model, WTM).  
- **Small-world structure** underlies “**six degrees of separation**” and differentially affects simple vs. complex contagion.  
- We combine simulation with mathematical heuristics (IC ↔ bond percolation; WTM ↔ vulnerable-cluster branching) and produce clear visuals for intuition.



## 1. Literature & Context

### What & Why
We anchor terminology and mechanisms in foundational research to clarify assumptions and typical use-cases (disease vs. behavior; reinforcement vs. single-shot exposure; identification challenges).

### 1.1 Six degrees & small worlds
- **Milgram (1967)**: surprisingly short social chains; popularized as “six degrees of separation.”
- **Watts & Strogatz (1998)**: **small-world networks** exhibit high clustering \(C\) with low average path length \(L\).  
  **Implication**: rapid reach with locally dense neighborhoods → key for reinforcement dynamics.

### 1.2 Network epidemiology & percolation
- **Newman (2002–2010)**: generating-function analyses of epidemics/percolation; thresholds governed by degree moments.
- **Pastor-Satorras et al. (2015)**: synthesis on epidemic processes and how heterogeneity alters thresholds.

### 1.3 Complex contagion & reinforcement
- **Centola & Macy (2007)**: “weakness of long ties” for behaviors requiring reinforcement.
- **Centola (2010)**: experiments showing clustering promotes behavior adoption.
- **Christakis & Fowler (2007–2013)**: empirical claims of social contagion (health, happiness, obesity, smoking), spurring debates about identification vs. homophily.

### 1.4 Why we care
- **Public health** (vaccination, misinformation); **innovation diffusion**; **systemic risk**; **policy design** (seeding & incentives).



## 2. Theory & Mathematical Foundations

### What & Why
We formalize how **structure** (degree distribution, clustering, path length) interacts with **process** (simple vs. complex contagion) to set cascade conditions and sizes.

### 2.1 Small-world metrics
Let \(L\) be mean geodesic distance, \(C\) the clustering coefficient. In WS graphs, increasing rewiring \(\beta\) sharply reduces \(L\) while \(C\) remains high for small \(\beta\), creating a **small-world regime** (high \(C\), low \(L\)).

### 2.2 Simple contagion (IC) ↔ bond percolation
For **Independent Cascade (IC)** on sparse networks, the final size from a seed corresponds to the component size in a **bond-percolated** graph with edge survival (transmissibility) \(T\approx p\). For a configuration model:
\[
R_0 \approx T \cdot \frac{\langle k^2\rangle - \langle k\rangle}{\langle k\rangle},
\]
so a **global cascade** requires \(R_0>1\). Degree heterogeneity lowers the threshold via \(\langle k^2\rangle\).

### 2.3 Complex contagion (WTM) & vulnerable clusters
In the **Watts Threshold Model (WTM)**, node \(i\) adopts if the adopted-neighbor fraction exceeds \(\phi_i\). Nodes with \(\phi_i<1/k_i\) are **vulnerable** (one neighbor suffices). Global cascades require **supercritical branching** among vulnerable nodes. **Clustering** enlarges vulnerable regions through reinforcement beyond tree-like approximations.

### 2.4 Competing roles of shortcuts and clustering
**Simple contagion**: shortcuts help (smaller \(L\)), clustering can slow.  
**Complex contagion**: clustering helps (redundant exposures), excessive shortcuts can hurt (destroy local triangles).



## 3. Setup

### What & Why
We load the scientific Python stack and set deterministic seeds for reproducibility. Visual defaults avoid styling so results remain comparable across environments.


In [ ]:

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from dataclasses import dataclass
from typing import Iterable, Optional, Dict, List, Union, Set

# Reproducibility and plotting
rng = np.random.default_rng(42)
plt.rcParams['figure.figsize'] = (6,4)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['axes.grid'] = True



## 4. Small Worlds & “Six Degrees” — Visualization

### What & Why
We quantify **average path length** \(L\) and **clustering** \(C\) across regimes, then visualize the distribution of geodesic distances to connect with the “six degrees” intuition.


In [ ]:

def ws_graph(n=300, k=6, beta=0.05, seed=None):
    k = k + (k % 2)
    return nx.watts_strogatz_graph(n, k, beta, seed=seed)

def summarize_graph(G: nx.Graph):
    return nx.average_shortest_path_length(G), nx.average_clustering(G)

def path_length_hist(G: nx.Graph, bins=None, samples=4000, rng=None):
    rng = rng or np.random.default_rng()
    nodes = list(G.nodes())
    m = min(samples, len(nodes)*(len(nodes)-1)//2)
    pairs = [(rng.integers(0, len(nodes)), rng.integers(0, len(nodes))) for _ in range(m)]
    dists = []
    for i,j in pairs:
        if i == j: 
            continue
        try:
            dists.append(nx.shortest_path_length(G, nodes[i], nodes[j]))
        except nx.NetworkXNoPath:
            pass
    if not dists:
        return
    if bins is None:
        bins = np.arange(0, max(dists)+2)
    plt.hist(dists, bins=bins, density=True)
    plt.xlabel("Shortest-path length")
    plt.ylabel("Density")
    plt.title("Geodesic length distribution")
    plt.show()

Gs = {
    "Lattice-like (β=0)":   ws_graph(n=250, k=6, beta=0.0, seed=rng),
    "Small-world (β=0.05)": ws_graph(n=250, k=6, beta=0.05, seed=rng),
    "Random-like (β=1)":    ws_graph(n=250, k=6, beta=1.0, seed=rng),
}

rows = []
for name, G in Gs.items():
    L, C = summarize_graph(G)
    rows.append(dict(model=name, L=L, C=C))
pd.DataFrame(rows)


In [ ]:

# Six-degrees-style distances in the small-world regime
path_length_hist(Gs["Small-world (β=0.05)"], rng=rng)



## 5. Contagion Models & Implementations

### What & Why
We implement two canonical processes with transparent APIs to facilitate replication and parameter sweeps:
- **Independent Cascade (IC)** — simple contagion (single effective exposure).
- **Watts Threshold Model (WTM)** — complex contagion (reinforcement).



### 5.1 Utilities (What & Why)
Seed selection strategies (random, degree, k-core) allow us to study **intervention design** (who to target). A `CascadeResult` container standardizes outputs for plotting/analysis.


In [ ]:

@dataclass
class CascadeResult:
    adopted: Set
    history: List[Set]
    adopters_by_t: List[int]
    exposures: Dict  # attempts (IC) or adopted-neighbor counts (WTM)
    steps: int

def choose_seeds(G: nx.Graph, k: int, method: str = "random", rng: Optional[np.random.Generator] = None) -> List:
    rng = rng or np.random.default_rng()
    nodes = list(G.nodes())
    if k <= 0: return []
    if method == "random":
        return list(rng.choice(nodes, size=min(k, len(nodes)), replace=False))
    if method == "degree":
        return [n for n,_ in sorted(G.degree(), key=lambda t: t[1], reverse=True)[:k]]
    if method == "kcore":
        kc = nx.core_number(G)
        return [n for n,_ in sorted(kc.items(), key=lambda t: t[1], reverse=True)[:k]]
    raise ValueError("Unknown seeding method")



### 5.2 Independent Cascade (IC) — What & Why
Models single-shot activation attempts per newly adopted node. Good proxy for **disease-like spread** or information that requires **one effective contact**.


In [ ]:

def simulate_ic(G: nx.Graph, seeds: Iterable, p: float = 0.1, max_steps: Optional[int] = None, rng: Optional[np.random.Generator] = None) -> CascadeResult:
    rng = rng or np.random.default_rng()
    adopted = set(seeds)
    frontier = set(seeds)
    history = [set(frontier)]
    adopters_by_t = [len(frontier)]
    exposures = {n:0 for n in G.nodes()}
    steps = 0
    while frontier and (max_steps is None or steps < max_steps):
        steps += 1
        new_frontier = set()
        for u in frontier:
            for v in G.neighbors(u):
                if v in adopted: 
                    continue
                exposures[v] = exposures.get(v,0) + 1
                if rng.random() < p:
                    adopted.add(v); new_frontier.add(v)
        if not new_frontier: break
        history.append(set(new_frontier))
        adopters_by_t.append(len(new_frontier))
        frontier = new_frontier
    return CascadeResult(adopted, history, adopters_by_t, exposures, steps)



### 5.3 Watts Threshold Model (WTM) — What & Why
Captures **reinforcement**—adoption requires a sufficient **fraction** of neighbors already adopted. Appropriate for behaviors, norms, or technologies that need social proof.


In [ ]:

def _resolve_thresholds(G: nx.Graph, phi) -> Dict:
    nodes = list(G.nodes())
    if isinstance(phi, (float,int)): return {n: float(phi) for n in nodes}
    if isinstance(phi, dict):        return {n: float(phi.get(n, 1.1)) for n in nodes}
    if callable(phi):                return {n: float(phi(n,G)) for n in nodes}
    arr = list(map(float, phi))
    if len(arr) != len(nodes): 
        raise ValueError("phi sequence length mismatch")
    return {n: arr[i] for i,n in enumerate(nodes)}

def simulate_wtm(G: nx.Graph, seeds: Iterable, phi=0.2, synchronous=True, max_steps: Optional[int] = None, rng: Optional[np.random.Generator] = None) -> CascadeResult:
    rng = rng or np.random.default_rng()
    thresholds = _resolve_thresholds(G, phi)
    adopted = set(seeds)
    history = [set(adopted)]
    adopters_by_t = [len(adopted)]
    steps = 0
    exposures = {n:0 for n in G.nodes()}  # adopted-neighbor count
    def frac(v):
        k = G.degree(v)
        if k == 0: return 0.0
        a = sum((1 for u in G.neighbors(v) if u in adopted))
        exposures[v] = a
        return a / k
    while True and (max_steps is None or steps < max_steps):
        steps += 1
        if synchronous:
            new_adopters = {v for v in G.nodes() if v not in adopted and frac(v) >= thresholds[v]}
        else:
            new_adopters = set()
            for v in rng.permutation(list(G.nodes())):
                if v in adopted or v in new_adopters: continue
                if frac(v) >= thresholds[v]:
                    new_adopters.add(v); adopted.add(v)
        if not new_adopters: break
        adopted |= new_adopters
        history.append(set(new_adopters))
        adopters_by_t.append(len(new_adopters))
    return CascadeResult(adopted, history, adopters_by_t, exposures, steps)



## 6. Visualizing Dynamics (IC vs WTM)

### What & Why
We compare **speed** and **extent** of adoption on the identical small-world graph to highlight how reinforcement changes outcomes.


In [ ]:

def plot_adoption_ts(res: CascadeResult, title: str = ""):
    y = np.cumsum(res.adopters_by_t)
    x = np.arange(len(y))
    plt.plot(x, y/len(res.exposures), marker="o", lw=1)
    plt.xlabel("Time"); plt.ylabel("Cumulative adoption (fraction)")
    plt.title(title); plt.ylim(0,1); plt.show()

G = ws_graph(n=300, k=6, beta=0.05, seed=rng)
seeds = choose_seeds(G, 5, method="degree", rng=rng)

res_ic  = simulate_ic(G, seeds, p=0.12, rng=rng)
res_wtm = simulate_wtm(G, seeds, phi=0.22, rng=rng)

plot_adoption_ts(res_ic,  "IC on small-world (p=0.12)")
plot_adoption_ts(res_wtm, "WTM on small-world (φ=0.22)")



## 7. Structure Experiments (ER vs BA vs WS)

### What & Why
We examine how **degree heterogeneity** (ER vs BA) and **clustering/shortcuts** (WS with rewiring \(\beta\)) alter cascade sizes under IC and WTM.


In [ ]:

def make_graph(kind: str = "ER", n: int = 400, avg_k: float = 6, beta: float = 0.1, rng=None) -> nx.Graph:
    rng = rng or np.random.default_rng()
    if kind.upper() == "ER":
        p = avg_k/(n-1); return nx.erdos_renyi_graph(n, p, seed=rng)
    if kind.upper() == "WS":
        k = int(round(avg_k)); k += (k%2); return nx.watts_strogatz_graph(n, k, beta, seed=rng)
    if kind.upper() == "BA":
        m = max(1, int(round(avg_k/2))); return nx.barabasi_albert_graph(n, m, seed=rng)
    raise ValueError

def sweep_ic(G, seeds, ps, runs=10, rng=None):
    rng = rng or np.random.default_rng()
    rows = []
    for p in ps:
        for r in range(runs):
            res = simulate_ic(G, seeds, p=p, rng=rng)
            rows.append(dict(p=p, frac=len(res.adopted)/G.number_of_nodes()))
    return pd.DataFrame(rows)

def sweep_wtm(G, seeds, phis, runs=1, rng=None):
    rng = rng or np.random.default_rng()
    rows = []
    for phi in phis:
        for r in range(runs):
            res = simulate_wtm(G, seeds, phi=phi, rng=rng)
            rows.append(dict(phi=phi, frac=len(res.adopted)/G.number_of_nodes()))
    return pd.DataFrame(rows)

G_er = make_graph("ER", n=350, avg_k=6, rng=rng)
G_ba = make_graph("BA", n=350, avg_k=6, rng=rng)
seeds_er = choose_seeds(G_er, 5, method="degree", rng=rng)
seeds_ba = choose_seeds(G_ba, 5, method="degree", rng=rng)

ps   = np.linspace(0.02, 0.25, 10)
phis = np.linspace(0.08, 0.5, 10)

df_er_ic = sweep_ic(G_er, seeds_er, ps, runs=6, rng=rng)
df_ba_ic = sweep_ic(G_ba, seeds_ba, ps, runs=6, rng=rng)
df_er_wt = sweep_wtm(G_er, seeds_er, phis, runs=1, rng=rng)
df_ba_wt = sweep_wtm(G_ba, seeds_ba, phis, runs=1, rng=rng)

plt.plot(df_er_ic.groupby("p")["frac"].mean(), marker="o", lw=1, label="ER")
plt.plot(df_ba_ic.groupby("p")["frac"].mean(), marker="o", lw=1, label="BA")
plt.xlabel("p (IC transmissibility)"); plt.ylabel("Final adoption (mean)")
plt.ylim(0,1); plt.title("IC: ER vs BA"); plt.legend(); plt.show()

plt.plot(df_er_wt.groupby("phi")["frac"].mean(), marker="o", lw=1, label="ER")
plt.plot(df_ba_wt.groupby("phi")["frac"].mean(), marker="o", lw=1, label="BA")
plt.xlabel("φ (WTM threshold)"); plt.ylabel("Final adoption (mean)")
plt.ylim(0,1); plt.title("WTM: ER vs BA"); plt.legend(); plt.show()


In [ ]:

# WS rewiring study: opposite sensitivities of IC vs WTM
betas = np.linspace(0.0, 1.0, 8)
rows_ic, rows_wt = [], []
for beta in betas:
    G_ws = make_graph("WS", n=300, avg_k=6, beta=beta, rng=rng)
    seeds_ws = choose_seeds(G_ws, 5, method="degree", rng=rng)
    res_ic = simulate_ic(G_ws, seeds_ws, p=0.12, rng=rng)
    res_wt = simulate_wtm(G_ws, seeds_ws, phi=0.22, rng=rng)
    rows_ic.append(dict(beta=beta, frac=len(res_ic.adopted)/G_ws.number_of_nodes()))
    rows_wt.append(dict(beta=beta, frac=len(res_wt.adopted)/G_ws.number_of_nodes()))

df_ws_ic = pd.DataFrame(rows_ic); df_ws_wt = pd.DataFrame(rows_wt)

plt.plot(df_ws_ic["beta"], df_ws_ic["frac"], marker="o", lw=1)
plt.xlabel("WS rewiring β"); plt.ylabel("Final adoption")
plt.ylim(0,1); plt.title("IC vs β (shortcuts help)"); plt.show()

plt.plot(df_ws_wt["beta"], df_ws_wt["frac"], marker="o", lw=1)
plt.xlabel("WS rewiring β"); plt.ylabel("Final adoption")
plt.ylim(0,1); plt.title("WTM vs β (reinforcement hindered by high β)"); plt.show()



## 8. Seeding Strategies (Dispersed vs Clustered)

### What & Why
We compare **random**, **high-degree**, and **k-core** seeding for both IC and WTM on a small-world graph, illustrating how **reinforcement** can favor **clustered** seeds in WTM.


In [ ]:

def compare_seeding(G, k=5, model="IC", param=0.1, rng=None):
    rng = rng or np.random.default_rng()
    rows = []
    for method in ["random", "degree", "kcore"]:
        seeds = choose_seeds(G, k, method=method, rng=rng)
        if model.upper() == "IC":
            res = simulate_ic(G, seeds, p=param, rng=rng)
        else:
            res = simulate_wtm(G, seeds, phi=param, rng=rng)
        rows.append(dict(method=method, frac=len(res.adopted)/G.number_of_nodes()))
    return pd.DataFrame(rows)

G_sw = make_graph("WS", n=300, avg_k=6, beta=0.05, rng=rng)
df_ic_seed  = compare_seeding(G_sw, k=6, model="IC",  param=0.12, rng=rng)
df_wtm_seed = compare_seeding(G_sw, k=6, model="WTM", param=0.22, rng=rng)

ax = df_ic_seed.set_index("method")["frac"].plot(kind="bar", rot=0)
ax.set_title("IC: Final adoption by seeding method"); ax.set_ylim(0,1); plt.show()

ax = df_wtm_seed.set_index("method")["frac"].plot(kind="bar", rot=0)
ax.set_title("WTM: Final adoption by seeding method"); ax.set_ylim(0,1); plt.show()



## 9. Analytical Touchpoints

### What & Why
We connect simulation outputs to parsimonious analytic heuristics that explain observed trends and guide prediction/design.

**IC ↔ percolation.** For sparse, locally tree-like graphs, final size for a single seed aligns with bond percolation at edge survival \(T\approx p\). Threshold heuristic:
\[
R_0 \approx T \cdot \frac{\langle k^2\rangle - \langle k\rangle}{\langle k\rangle} > 1.
\]

**WTM vulnerable clusters.** Nodes with \(\phi<1/k\) are vulnerable; global cascades require supercritical branching of vulnerable nodes. **Clustering** expands cascade regimes via reinforcement beyond tree-like predictions.



## 10. Exercises & Extensions

1. **Temporal reinforcement window:** adopt if ≥ \(m\) exposures within \(W\) steps. Explore tradeoffs among \(m\), \(W\), and clustering.
2. **Weighted/directed thresholds:** adopt if weighted neighbor fraction ≥ \(\phi\); experiment with in-neighbor-only dynamics on directed graphs.
3. **Multiplex reinforcement:** layers \(A,B\); adopt if \(f_A \ge \phi_A\) **or** \(f_A + f_B \ge \Phi\).
4. **Causal identification:** outline designs to separate contagion from homophily (randomized seeding, IVs, panel models with network fixed effects).



## 11. References (select)
- Milgram, S. (1967). *The small-world problem*. Psychology Today.  
- Watts, D. J., & Strogatz, S. H. (1998). *Collective dynamics of ‘small-world’ networks*. Nature.  
- Newman, M. E. J. (2002–2010). Random graphs & networks; percolation & epidemics on networks.  
- Pastor-Satorras, R., Castellano, C., Van Mieghem, P., & Vespignani, A. (2015). *Epidemic processes in complex networks*. Rev. Mod. Phys.  
- Kempe, D., Kleinberg, J., & Tardos, É. (2003). *Maximizing the spread of influence through a social network*. KDD.  
- Watts, D. J. (2002). *A simple model of global cascades on random networks*. PNAS.  
- Centola, D., & Macy, M. (2007). *Complex contagions and the weakness of long ties*. AJS.  
- Centola, D. (2010). *The spread of behavior in an online social network experiment*. Science.  
- Christakis, N. A., & Fowler, J. H. (2007–2013). Social contagion in health and behavior.
